In [ ]:
'''
Machine Learning using Python and PySpark.
Dataset:  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

Some of the concepts covered here:
 - DataFrames: Distributed data handling.
 - MLlib Pipelines: Chain preprocessing + model.
 - Feature Engineering: StringIndexer, OneHotEncoder, VectorAssembler.
 - Supervised Learning: Logistic regression, classification
 - Performance evaluation: Confusion matrix
 - Scalability: This code works the same on massive clusters.
 '''

In [14]:
# Get the libraries

!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pyspark.sql.functions as F

# Create Spark Session
spark = SparkSession.builder \
    .appName("PySpark ML Titanic") \
    .getOrCreate()

print("✅ Spark Session Created Successfully!")

✅ Spark Session Created Successfully!


In [15]:
# Load the dataset
import requests

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
local_path = "titanic.csv"

# Download file to local storage
response = requests.get(url)
with open(local_path, "wb") as f:
    f.write(response.content)

print("✅ Dataset downloaded successfully!")

# Read from local file
df = spark.read.csv(local_path, header=True, inferSchema=True)

df.show(5)
df.printSchema()

✅ Dataset downloaded successfully!
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+----------------

In [16]:
# Data exploration and cleaning
# Basic stats
df.describe().show()

# Handle missing values
df = df.na.drop()

# Select relevant columns
df = df.select(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'])

print ("Data exploration and cleaning: completed")

+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|summary|      PassengerId|           Survived|            Pclass|                Name|   Sex|               Age|             SibSp|              Parch|            Ticket|             Fare|Cabin|Embarked|
+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|  count|              891|                891|               891|                 891|   891|               714|               891|                891|               891|              891|  204|     889|
|   mean|            446.0| 0.3838383838383838| 2.308641975308642|                NULL|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824|260318.54916792738| 32.20420

In [17]:
# Feature engineering
# Index categorical columns
indexer_sex = StringIndexer(inputCol="Sex", outputCol="SexIndex")
indexer_emb = StringIndexer(inputCol="Embarked", outputCol="EmbarkedIndex")

# One-hot encode
encoder = OneHotEncoder(inputCols=["SexIndex", "EmbarkedIndex"],
                       outputCols=["SexVec", "EmbarkedVec"])

# Assemble features
assembler = VectorAssembler(inputCols=['Pclass', 'Age', 'SibSp', 'Parch', 'Fare',
                                      'SexVec', 'EmbarkedVec'],
                           outputCol="features")

# Label column is already 'Survived'
print ("Label column ready")

Label column ready


In [18]:
# Build pipeline and split the data
# Pipeline
pipeline = Pipeline(stages=[indexer_sex, indexer_emb, encoder, assembler])

# Transform data
transformed = pipeline.fit(df).transform(df)

# Split
train, test = transformed.randomSplit([0.8, 0.2], seed=42)

print ("Pipeline ready. Data split.")

Pipeline ready. Data split.


In [19]:
# Train the model:  Logistic regression
lr = LogisticRegression(featuresCol="features", labelCol="Survived",
                       predictionCol="prediction", probabilityCol="probability")

model = lr.fit(train)

# Predictions
predictions = model.transform(test)
predictions.select("Survived", "prediction", "probability").show(10)

print("Logistic regression complete")

+--------+----------+--------------------+
|Survived|prediction|         probability|
+--------+----------+--------------------+
|       0|       1.0|[0.06043318031326...|
|       0|       1.0|[0.40688766912561...|
|       0|       1.0|[0.31146979127709...|
|       0|       1.0|[0.41081299511057...|
|       0|       0.0|[0.50233509249317...|
|       0|       1.0|[5.02416464074937...|
|       0|       0.0|[0.60345935352224...|
|       0|       0.0|[0.64989449608305...|
|       0|       0.0|[0.74804349723162...|
|       0|       0.0|[0.75917616788566...|
+--------+----------+--------------------+
only showing top 10 rows
Logistic regression complete


In [20]:
# Evaluate the model
evaluator = BinaryClassificationEvaluator(labelCol="Survived",
                                         rawPredictionCol="rawPrediction",
                                         metricName="areaUnderROC")

auc = evaluator.evaluate(predictions)
print(f"Area Under ROC: {auc}")

# Confusion Matrix (simple)
predictions.groupBy("Survived", "prediction").count().show()

print("Evaluation complete")

Area Under ROC: 0.7666666666666667
+--------+----------+-----+
|Survived|prediction|count|
+--------+----------+-----+
|       1|       0.0|    2|
|       0|       0.0|    8|
|       1|       1.0|   13|
|       0|       1.0|    6|
+--------+----------+-----+

Evaluation complete


In [24]:
# Save the model and predictions: CSV file
# Save the trained model
model.write().overwrite().save("titanic_lr_model")
print("✅ Model saved successfully!")

# Convert vectors to readable format
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import array_join, col

predictions_for_save = predictions.select(
    col("Survived").cast("int").alias("Survived"),
    col("prediction").cast("int").alias("prediction"),
    array_join(vector_to_array("probability"), ", ").alias("probability")   # Fixed
)

predictions_for_save.write.csv("titanic_predictions.csv",
                               header=True,
                               mode="overwrite")

print("✅ Predictions saved successfully to titanic_predictions.csv")

# Show sample
predictions_for_save.show(10, truncate=False)

✅ Model saved successfully!
✅ Predictions saved successfully to titanic_predictions.csv
+--------+----------+-----------------------------------------+
|Survived|prediction|probability                              |
+--------+----------+-----------------------------------------+
|0       |1         |0.06043318031326829, 0.9395668196867317  |
|0       |1         |0.4068876691256116, 0.5931123308743884   |
|0       |1         |0.3114697912770933, 0.6885302087229067   |
|0       |1         |0.41081299511057945, 0.5891870048894206  |
|0       |0         |0.5023350924931744, 0.49766490750682557  |
|0       |1         |5.0241646407493774E-8, 0.9999999497583536|
|0       |0         |0.6034593535222407, 0.39654064647775933  |
|0       |0         |0.6498944960830522, 0.35010550391694784  |
|0       |0         |0.7480434972316282, 0.2519565027683718   |
|0       |0         |0.759176167885668, 0.240823832114332     |
+--------+----------+-----------------------------------------+
only showing top